In [3]:
from pathlib import Path
import pandas as pd
import numpy as np


base = Path("../../data/full")
features = ["ra", "dec", "parallax", "pmra", "pmdec", "phot_g_mean_mag"]


def pick_existing(cands, name):
    fp = next((p for p in cands if p.exists()), None)
    assert fp is not None, f"{name} not found. Checked: {cands}"
    return fp

def read_csv_safe(fp):
    try:
        return pd.read_csv(fp)
    except Exception:
        return pd.read_csv(fp, engine="python", on_bad_lines="skip")

# robust alias map for exoplanet schemas
aliases = {
    "ra": ["ra", "ra_deg", "gaia_ra"],
    "dec": ["dec", "dec_deg", "gaia_dec"],
    "parallax": ["parallax", "sy_plx", "plx"],
    "pmra": ["pmra", "sy_pmra"],
    "pmdec": ["pmdec", "sy_pmdec"],
    "phot_g_mean_mag": ["phot_g_mean_mag", "sy_gaiamag", "gmag", "gaia_gmag"],
}

def harmonize_exo(df):
    d = df.copy()
    col_lut = {c.lower(): c for c in d.columns}
    rename = {}
    for target, cands in aliases.items():
        for cand in cands:
            if cand.lower() in col_lut:
                rename[col_lut[cand.lower()]] = target
                break
    d = d.rename(columns=rename)
    return d

def prep_shared(df, is_exo=False):
    d = harmonize_exo(df) if is_exo else df.copy()
    for c in features:
        if c not in d.columns:
            d[c] = np.nan
    d[features] = d[features].apply(pd.to_numeric, errors="coerce")
    return d

def counts(raw_df, clean_df_p):
    rows_raw = len(raw_df)
    rows_after_script = len(clean_df_p)

    missing_cells_before = int(clean_df_p[features].isna().sum().sum())

    complete_df = clean_df_p.dropna(subset=features)
    rows_after_dropna = len(complete_df)
    missing_cells_after = int(complete_df[features].isna().sum().sum())  # should be 0

    removed_rows = rows_after_script - rows_after_dropna
    removed_pct = (removed_rows / rows_after_script * 100) if rows_after_script else 0.0

    return (
        rows_raw,
        rows_after_script,
        rows_after_dropna,
        removed_rows,
        removed_pct,
        missing_cells_before,
        missing_cells_after,
    )


stars_raw_fp   = pick_existing([base / "stars_gaia.csv", base / "stars.csv"], "stars raw")
stars_clean_fp = pick_existing([base / "stars_gaia_clean.csv", base / "stars_clean.csv"], "stars clean")

quas_raw_fp    = pick_existing([base / "quasars_gaia.csv", base / "quasars.csv"], "quasars raw")
quas_clean_fp  = pick_existing([base / "quasars_gaia_clean.csv", base / "quasars_clean.csv"], "quasars clean")

# for exoplanets, choose the candidate with best complete rows on shared features
exo_candidates = [base / "exoplanets_clean.csv", base / "exoplanets_gaia_enriched.csv"]
exo_candidates = [p for p in exo_candidates if p.exists()]
assert exo_candidates, "No exoplanet candidate files found."

best = None
for fp in exo_candidates:
    d_raw = read_csv_safe(fp)
    d_prep = prep_shared(d_raw, is_exo=True)
    complete_rows = d_prep.dropna(subset=features).shape[0]
    non_null = d_prep[features].notna().sum().to_dict()
    print(f"{fp.name}: complete_rows={complete_rows}, non_null={non_null}")
    if best is None or complete_rows > best["complete_rows"]:
        best = {"fp": fp, "raw": d_raw, "prep": d_prep, "complete_rows": complete_rows}

exo_raw_fp = best["fp"]

print("\nUsing files:")
print(" stars raw      :", stars_raw_fp)
print(" stars clean    :", stars_clean_fp)
print(" quasars raw    :", quas_raw_fp)
print(" quasars clean  :", quas_clean_fp)
print(" exo selected   :", exo_raw_fp)


stars_raw = read_csv_safe(stars_raw_fp)
stars_clean = read_csv_safe(stars_clean_fp)
stars_p = prep_shared(stars_clean, is_exo=False)

quas_raw = read_csv_safe(quas_raw_fp)
quas_clean = read_csv_safe(quas_clean_fp)
quas_p = prep_shared(quas_clean, is_exo=False)

exo_raw = best["raw"]      # selected best candidate as raw baseline
exo_p = best["prep"]       # already harmonized + numeric coerced

print("\nExoplanet non-null counts in shared features:")
print(exo_p[features].notna().sum())


s = counts(stars_raw, stars_p)
q = counts(quas_raw, quas_p)
e = counts(exo_raw, exo_p)

table3_counts = pd.DataFrame([
    {"Dataset": "Stars", "Rows (raw)": s[0], "Rows (after script cleaning)": s[1], "Rows (after shared-feature dropna)": s[2]},
    {"Dataset": "Quasars", "Rows (raw)": q[0], "Rows (after script cleaning)": q[1], "Rows (after shared-feature dropna)": q[2]},
    {"Dataset": "Exoplanet hosts", "Rows (raw)": e[0], "Rows (after script cleaning)": e[1], "Rows (after shared-feature dropna)": e[2]},
])

table5_missing = pd.DataFrame([
    {"Dataset": "Stars", "Missing cells before": s[5], "Missing cells after": s[6], "Rows removed by dropna": s[3], "Rows removed (%)": round(s[4], 2)},
    {"Dataset": "Quasars", "Missing cells before": q[5], "Missing cells after": q[6], "Rows removed by dropna": q[3], "Rows removed (%)": round(q[4], 2)},
    {"Dataset": "Exoplanet hosts", "Missing cells before": e[5], "Missing cells after": e[6], "Rows removed by dropna": e[3], "Rows removed (%)": round(e[4], 2)},
])

print("\nTable 3 counts")
display(table3_counts)

print("Table 5 missing-handling")
display(table5_missing)


t3 = table3_counts.copy()
for c in ["Rows (raw)", "Rows (after script cleaning)", "Rows (after shared-feature dropna)"]:
    t3[c] = t3[c].astype(int)

t5 = table5_missing.copy()
for c in ["Missing cells before", "Missing cells after", "Rows removed by dropna"]:
    t5[c] = t5[c].astype(int)
t5["Rows removed (%)"] = t5["Rows removed (%)"].round(2)

latex_t3 = t3.to_latex(
    index=False,
    caption="Cleaning-stage row counts per dataset.",
    label="tab:cleaning_steps_counts",
    column_format="lrrr",
    escape=True
)

latex_t5 = t5.to_latex(
    index=False,
    caption="Missing-value handling summary before and after shared-feature filtering.",
    label="tab:missing_before_after",
    column_format="lrrrr",
    escape=True
)

print("\n--- LaTeX: Table 3 ---")
print(latex_t3)
print("\n--- LaTeX: Table 5 ---")
print(latex_t5)


exoplanets_clean.csv: complete_rows=37871, non_null={'ra': 37871, 'dec': 37871, 'parallax': 37871, 'pmra': 37871, 'pmdec': 37871, 'phot_g_mean_mag': 37871}
exoplanets_gaia_enriched.csv: complete_rows=0, non_null={'ra': 37871, 'dec': 37871, 'parallax': 37871, 'pmra': 0, 'pmdec': 0, 'phot_g_mean_mag': 0}

Using files:
 stars raw      : ..\..\data\full\stars_gaia.csv
 stars clean    : ..\..\data\full\stars_gaia_clean.csv
 quasars raw    : ..\..\data\full\quasars_gaia.csv
 quasars clean  : ..\..\data\full\quasars_gaia_clean.csv
 exo selected   : ..\..\data\full\exoplanets_clean.csv

Exoplanet non-null counts in shared features:
ra                 37871
dec                37871
parallax           37871
pmra               37871
pmdec              37871
phot_g_mean_mag    37871
dtype: int64

Table 3 counts


,Dataset,Rows (raw),Rows (after script cleaning),Rows (after shared-feature dropna)
0,Stars,50000,50000,50000
1,Quasars,50000,50000,50000
2,Exoplanet hosts,37871,37871,37871


Table 5 missing-handling


,Dataset,Missing cells before,Missing cells after,Rows removed by dropna,Rows removed (%)
0,Stars,0,0,0,0.0
1,Quasars,0,0,0,0.0
2,Exoplanet hosts,0,0,0,0.0



--- LaTeX: Table 3 ---
\begin{table}
\caption{Cleaning-stage row counts per dataset.}
\label{tab:cleaning_steps_counts}
\begin{tabular}{lrrr}
\toprule
Dataset & Rows (raw) & Rows (after script cleaning) & Rows (after shared-feature dropna) \\
\midrule
Stars & 50000 & 50000 & 50000 \\
Quasars & 50000 & 50000 & 50000 \\
Exoplanet hosts & 37871 & 37871 & 37871 \\
\bottomrule
\end{tabular}
\end{table}


--- LaTeX: Table 5 ---
\begin{table}
\caption{Missing-value handling summary before and after shared-feature filtering.}
\label{tab:missing_before_after}
\begin{tabular}{lrrrr}
\toprule
Dataset & Missing cells before & Missing cells after & Rows removed by dropna & Rows removed (\%) \\
\midrule
Stars & 0 & 0 & 0 & 0.000000 \\
Quasars & 0 & 0 & 0 & 0.000000 \\
Exoplanet hosts & 0 & 0 & 0 & 0.000000 \\
\bottomrule
\end{tabular}
\end{table}

